# 8.1 Data Preprocessing

**Goal:** fit preprocessing on training data and distinguish missingness, scaling, and outlier policies.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Data Preprocessing](1_Data-Preprocessing.ipynb)
2. [Feature Engineering](2_Feature-Engineering.ipynb)
3. [Model Evaluation Statistics](3_Model-Evaluation-Statistics.ipynb)
4. [Statistical Learning Concepts](4_Statistical-Learning-Concepts.ipynb)
5. [Advanced ML Statistics](5_Advanced-ML-Statistics.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Missing Data Mechanisms: MCAR, MAR, MNAR

**MCAR** (missing completely at random) means missingness is independent of observed and missing values under the model. **MAR** means missingness can depend on observed information but, conditional on it, not the missing value. **MNAR** means dependence on unobserved values remains after conditioning on observed information.

These are assumptions about the missingness process, not labels that can usually be proven from a spreadsheet. For example, income nonresponse depending on income even after observed covariates is MNAR. Observed age explaining response can make a MAR model plausible, but cannot rule out residual MNAR. Missingness itself can be predictive, yet may reflect collection artifacts that change at deployment.

## 2. Mean and Median Imputation; Mode Imputation

Mean imputation fills a numeric variable with its observed training mean; median imputation uses its median; mode imputation fills a category with a most frequent observed level. Handle ties and all-missing features explicitly. Missing categories may deserve an explicit unknown level rather than a falsely certain mode.

These single imputations suppress uncertainty and can distort variance, correlations, and inferential SEs. Median resists extreme magnitudes more than mean. Multiple imputation targets a different inferential task and needs a model. Fit imputation values within training folds and apply unchanged to validation/test; never let future values determine fill statistics.

In [2]:
train=np.array([[1.,10.],[2.,np.nan],[3.,12.],[100.,14.]])
mean_fill=np.nanmean(train,axis=0);median_fill=np.nanmedian(train,axis=0)
mean_imputed=np.where(np.isnan(train),mean_fill,train)
median_imputed=np.where(np.isnan(train),median_fill,train)
print("Training means/medians:",mean_fill,median_fill)
print("Median-imputed rows:",median_imputed)

Training means/medians: [26.5 12. ] [ 2.5 12. ]
Median-imputed rows: [[  1.  10.]
 [  2.  12.]
 [  3.  12.]
 [100.  14.]]


## 3. KNN Imputation

K-nearest-neighbor imputation fills from nearby training records with observed donor values. Distances should use suitable scaled features and a stated policy for partially overlapping observations. A row with no comparable observed features needs a fallback; a donor cannot provide a missing value it also lacks. K, weighting, and feature choice affect results.

**Example:** impute the missing second feature using two nearest donors by the fully observed first feature. This simple one-distance-coordinate example is not a general mixed-data missing-aware distance implementation. Donor search must remain training-only when transforming held-out records.

In [3]:
target=1;donors=np.flatnonzero(~np.isnan(train[:,1]))
distances=np.abs(train[donors,0]-train[target,0]);nearest=donors[np.argsort(distances)[:2]]
print("Nearest donor IDs:",nearest,"KNN fill:",train[nearest,1].mean())

Nearest donor IDs: [0 2] KNN fill: 11.0


## 4. Outlier Treatment

Investigate source, units, duplicates, subgroup structure, and validity before deleting/capping an outlier. Options include keeping it, correcting a documented error, a justified exclusion, robust losses, transformations, or winsorizing at training-fitted quantiles. Rare fraud or high-cost cases can be essential, so automatic deletion changes the target population.

## 5. Normalization, Standardization, and Robust Scaling

“Normalization” is ambiguous: min-max scaling maps training endpoints to [0,1], while vector normalization divides each row by an L1/L2 norm. Standardization uses training (x−mean)/SD; robust scaling often uses (x−median)/IQR. None guarantees normality. Zero ranges, zero SD/IQR, and zero row norms need a policy. Held-out min-max values can exceed [0,1].

Scaling matters for distance and penalty-based models; tree splits often have different sensitivities. Fitting scales on all data leaks held-out information even without labels.

In [4]:
x=median_imputed
sd=x.std(0);iqr=np.quantile(x,.75,axis=0)-np.quantile(x,.25,axis=0)
standard=(x-x.mean(0))/np.where(sd>0,sd,1)
robust=(x-np.median(x,axis=0))/np.where(iqr>0,iqr,1)
minimum,span=x.min(0),np.ptp(x,axis=0)
minmax=(x-minimum)/np.where(span>0,span,1)
row_norm=np.linalg.norm(x,axis=1,keepdims=True);unit=x/np.where(row_norm>0,row_norm,1)
print("Standardized mean/SD:",standard.mean(0),standard.std(0))
print("Robust scaled first feature:",robust[:,0],"row L2 norms:",np.linalg.norm(unit,axis=1))
assert np.allclose(standard.mean(0),0)

Standardized mean/SD: [0. 0.] [1. 1.]
Robust scaled first feature: [-0.05882353 -0.01960784  0.01960784  3.82352941] row L2 norms: [1. 1. 1. 1.]


## 6. Distribution Transformation

Log/log1p can compress right-skewed nonnegative values; Box–Cox requires positive values and estimates a power; Yeo–Johnson can accommodate nonpositive inputs. Quantile transforms map empirical ranks toward a chosen reference but can distort distances and tail extrapolation. Fitted transformation parameters and rank maps must come from training data.

**Worked example:** log1p retains a zero amount and compresses 100 relative to 1, but the resulting units are log-transformed and back-transforming a mean prediction is not generally an unbiased original-scale mean. Transformation choice belongs inside model validation.

In [5]:
amounts=np.array([0.,1.,2.,4.,10.,100.])
print("Original amounts:",amounts,"log1p:",np.log1p(amounts))
transformed,lam=stats.yeojohnson(np.array([-2.,-1.,0.,1.,4.,10.]))
print("Yeo-Johnson fitted lambda:",lam)

Original amounts: [  0.   1.   2.   4.  10. 100.] log1p: [0.         0.69314718 1.09861229 1.60943791 2.39789527 4.61512052]
Yeo-Johnson fitted lambda: 0.39612509497078613


## AI/ML application and interpretation

Build one training-only pipeline for imputation, encoding, scaling, and model fitting. Refit that whole pipeline within each validation fold. Audit collection-time availability and deployment missingness patterns; preprocessing can be technically correct yet rely on features unavailable when predictions are made.

## Practice

1. Can an observed association between age and missing income prove MAR?
2. Why can mean imputation distort correlation?
3. What training quantity should scale a held-out value?
4. Does log1p guarantee a normal feature?

## Solutions

1. No. It can support a missingness model but cannot rule out remaining dependence on unobserved income.
2. It inserts repeated constants, changes variance and joint relationships, and treats uncertainty as a known value.
3. The training-fitted mean/SD, median/IQR, range, or other chosen transform; never refit on the held-out value.
4. No. It changes geometry and can reduce skew without establishing normality.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](2_Feature-Engineering.ipynb).